In [1]:
import subprocess
import time
import pandas as pd

input_file ="eng_updated selected dataset.xlsx"
output_file ="english_to_hindi_results.xlsx"

ID_COL   = "sentence_id"
TEXT_COL = "orginal_sen"

MODEL = "translategemma:4b"


In [2]:
def translate_en_to_hi(text):

    prompt = (
        "Translate the following English sentence to Hindi.\n"
        "Return only the Hindi translation.\n\n"
        f"{text}"
    )

    result = subprocess.run(
        ["ollama", "run", MODEL],
        input=prompt,
        capture_output=True,
        text=True,
        encoding="utf-8"
    )

    return result.stdout.strip()

In [3]:
print(translate_en_to_hi("my name is bhavana."))

मेरा नाम भवानी है।


In [4]:
df = pd.read_excel(input_file)
df.columns = df.columns.str.strip()

df = df[[ID_COL, TEXT_COL]].dropna().copy()

print("Rows loaded:", len(df))
df.head()


Rows loaded: 200


,sentence_id,orginal_sen
0,1,Let's try something.
1,2,I have to go to sleep.
2,3,Today is June 18th and it is Muiriel's birthday!
3,4,Muiriel is 20 now.
4,5,"The password is ""Muiriel""."


In [5]:
preview = df.head(4).copy()
translated_preview =[]

for text in preview[TEXT_COL]:
    hi = translate_en_to_hi(text)
    translated_preview.append(hi)
    time.sleep(0.3)
preview["translated_hi"] = translated_preview
preview

,sentence_id,orginal_sen,translated_hi
0,1,Let's try something.,"चलिए, हम कुछ कोशिश करते हैं।"
1,2,I have to go to sleep.,मुझे सोने जाना है।
2,3,Today is June 18th and it is Muiriel's birthday!,आज 18 जून है और यह मियरील का जन्मदिन है!
3,4,Muiriel is 20 now.,म्युरियल अब 20 साल की है।


In [6]:
translate_all =[]
for text in df[TEXT_COL]:
    hi = translate_en_to_hi(text)
    translate_all.append(hi)
df["translate_hi"]=translate_all
import re

def clean_text(text):
    if isinstance(text, str):
        return re.sub(r"[\x00-\x1F\x7F]", "", text)
    return text

df = df.applymap(clean_text)
df.to_excel(output_file, index=False)
print(output_file) 


english_to_hindi_results.xlsx


C:\Users\manda\AppData\Local\Temp\ipykernel_33104\3313277342.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_text)


In [7]:
def translate_hi_to_en(text):
    
    prompt = (
        "Translate the following Hindi sentence to English.\n"
        "Return only the English translation.\n\n"
        f"{text}"
    )

    result = subprocess.run(
        ["ollama", "run", MODEL],
        input=prompt,
        capture_output=True,
        text=True,
        encoding="utf-8"
    )
    return result.stdout.strip()

In [8]:
back_translation = []
for text in df["translate_hi"]:
    en_back = translate_hi_to_en(text)
    back_translation.append(en_back)
    time.sleep(0.3)
df["back_translation_en"] = back_translation
df.head()

,sentence_id,orginal_sen,translate_hi,back_translation_en
0,1,Let's try something.,चलो कुछ करके देखते हैं।,Let's see what we can do.
1,2,I have to go to sleep.,मुझे सोने जाना है।,I have to go to sleep.
2,3,Today is June 18th and it is Muiriel's birthday!,आज 18 जून है और यह मियरील का जन्मदिन है!,"Today is June 18th, and it's Mireil's birthday!"
3,4,Muiriel is 20 now.,म्युरियल अब 20 साल की है।,Myriel is now 20 years old.
4,5,"The password is ""Muiriel"".","पासवर्ड ""म्युरिएल"" है।","The password is ""Myriel""."


In [9]:
import re

def clean_text(text):
    if isinstance(text, str):
        return re.sub(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]", "", text)
    return text

df = df.applymap(clean_text)

C:\Users\manda\AppData\Local\Temp\ipykernel_33104\2977540570.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_text)


In [10]:
df.to_excel("english_hindi_backtranslation_results.xlsx", index=False)
